# 08. Preprocesamiento de Datos y Preparación para el Modelado

Este notebook corresponde a la **Fase 3: Tarea 3.1 - Preprocesamiento**. 
En esta etapa prepararemos la parrilla de salida para nuestros modelos predictivos de Inteligencia Artificial (XGBoost y LightGBM).

### Justificación Metodológica: ¿Por qué no normalizamos ni escalamos?
A diferencia de los modelos lineales o basados en distancias (como SVM o K-NN), los algoritmos basados en árboles de decisión (como Random Forest, XGBoost y LightGBM) son **invariantes ante transformaciones monótonas** de las variables de entrada. 
Esto significa que dividir una variable por su máximo o restarle su media no altera los puntos de corte óptimos que el árbol encuentra para segmentar los datos.

Evitando el escalado (como *StandardScaler* o *MinMaxScaler*):
1. **Preservamos la interpretabilidad física directa:** Las variables retienen sus unidades físicas originales (grados de pendiente, metros de altitud, km/h de velocidad de viento, ºC de temperatura). Esto es extremadamente valioso para la memoria del TFG y la posterior interpretabilidad del modelo (SHAP).
2. **Evitamos riesgo de data leakage:** No introducimos estadísticas globales de test en el conjunto de entrenamiento.
3. **Eficiencia y sencillez:** Menos pasos en el pipeline de preprocesamiento.

## 1. Carga de Librerías y Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path


BASE_DIR = Path("..")
DATA_PATH = BASE_DIR / "datos" / "processed" / "08_dataset_modelado_BALANCEADO.csv"
OUT_DIR = BASE_DIR / "datos" / "processed"

print(f"Cargando dataset balanceado desde: {DATA_PATH.name}...")
df = pd.read_csv(DATA_PATH, sep=';', encoding='utf-8')

print(f"Dimensiones iniciales del dataset: {df.shape[0]} filas y {df.shape[1]} columnas")

Cargando dataset balanceado desde: 08_dataset_modelado_BALANCEADO.csv...
Dimensiones iniciales del dataset: 8952 filas y 20 columnas


## 2. Limpieza y Homogeneización de Tipos de Datos

Para asegurar que todas las variables numéricas sean interpretadas correctamente por Python como números reales (float), normalizaremos cualquier posible separador decimal de tipo europeo (coma) a punto.

In [2]:

cols_numericas = [
    'lat', 'lon', 'Superficie_Total_Real', 'elevacion', 'pendiente', 'orientacion',
    'temp_max', 'temp_media', 'temp_min', 'precipitacion', 'viento_medio', 'racha_max',
    'dir_viento', 'humedad_media', 'prec_acum_7d', 'tmax_max_7d', 'dias_sin_lluvia'
]

for col in cols_numericas:
    if col in df.columns:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce')
        else:
            df[col] = pd.to_numeric(df[col], errors='coerce')

print("Tipos de datos actualizados y validados:")
print(df[cols_numericas].dtypes)

Tipos de datos actualizados y validados:
lat                      float64
lon                      float64
Superficie_Total_Real    float64
elevacion                float64
pendiente                float64
orientacion              float64
temp_max                 float64
temp_media               float64
temp_min                 float64
precipitacion            float64
viento_medio             float64
racha_max                float64
dir_viento               float64
humedad_media            float64
prec_acum_7d             float64
tmax_max_7d              float64
dias_sin_lluvia          float64
dtype: object


## 3. Codificación de Variables Categóricas

### 3.1. One-Hot Encoding de 'tipo_vegetacion'
Utilizaremos One-Hot Encoding (`pd.get_dummies`) sobre la columna `tipo_vegetacion` manteniendo `drop_first=False` para generar columnas binarias limpias e independientes. Esto facilitará posteriormente el análisis de la importancia de variables con **SHAP (SHapley Additive exPlanations)**, ya que cada clase forestal y de cultivo tendrá su propia variable predictora directa sin estar oculta en la variable omitida.

Primero, normalizaremos y limpiaremos los nombres de las categorías forestales.

In [3]:

df['tipo_vegetacion'] = df['tipo_vegetacion'].astype(str).str.strip()
df['tipo_vegetacion'] = df['tipo_vegetacion'].str.replace(r'.*Con.*feras.*', 'Coniferas', regex=True)
df['tipo_vegetacion'] = df['tipo_vegetacion'].str.replace(r'.*Agr.*cola.*', 'Agricola', regex=True)
df['tipo_vegetacion'] = df['tipo_vegetacion'].str.replace(r'.*Matorral.*', 'Matorral', regex=True)
df['tipo_vegetacion'] = df['tipo_vegetacion'].str.replace(r'.*Pastizal.*', 'Pastizal', regex=True)
df['tipo_vegetacion'] = df['tipo_vegetacion'].str.replace(r'.*Frondosas.*', 'Frondosas', regex=True)
df['tipo_vegetacion'] = df['tipo_vegetacion'].str.replace(r'.*Urbano/Otros.*', 'Urbano_Otros', regex=True)


print("Categorías normalizadas:")
print(df['tipo_vegetacion'].value_counts())


df_dummies = pd.get_dummies(df['tipo_vegetacion'], prefix='veg', prefix_sep='_', drop_first=False)


df = pd.concat([df.drop(columns=['tipo_vegetacion']), df_dummies], axis=1)

print(f"\nNuevas columnas de vegetación agregadas:")
print([col for col in df.columns if col.startswith('veg_')])

Categorías normalizadas:
tipo_vegetacion
Coniferas             2650
Matorral              2246
Agricola              1973
Frondosas              874
Urbano_Otros           698
Pastizal               504
Urbano_Antropizado       7
Name: count, dtype: int64

Nuevas columnas de vegetación agregadas:
['veg_Agricola', 'veg_Coniferas', 'veg_Frondosas', 'veg_Matorral', 'veg_Pastizal', 'veg_Urbano_Antropizado', 'veg_Urbano_Otros']


### 3.2. Codificación de 'dir_viento'
La columna `dir_viento` almacena la dirección del viento dominante. Validamos si se encuentra en formato numérico continuo (grados de 0 a 360) o requiere transformación.

In [4]:
if df['dir_viento'].dtype == 'object':
    print("La columna 'dir_viento' contiene texto cardinal. Aplicando codificación categórica...")
    df['dir_viento'] = pd.Categorical(df['dir_viento']).codes
else:
    print("La columna 'dir_viento' ya es numérica y continua (grados de 0 a 360). Se conserva intacta.")

La columna 'dir_viento' ya es numérica y continua (grados de 0 a 360). Se conserva intacta.


## 4. Eliminación de Variables Sesgadas / Chivatas (Leakage)

Para garantizar la validez científica y el comportamiento del modelo predictivo en tiempo real, debemos eliminar variables que den una pista directa de la clase objetivo (target) o que no aporten valor predictivo en tiempo real:
1. **`Superficie_Total_Real`:** Esta variable es una **chivata extrema (data leakage)**. Para todas las ausencias (target=0) su valor es `0.0`, mientras que para los incendios reales toma el valor de la superficie quemada. Si la dejáramos, la Inteligencia Artificial aprendería el truco trampa de clasificar como incendio cualquier fila con superficie > 0, invalidando la utilidad del modelo.
2. **`fecha_ini`:** Es un campo de fecha y hora en formato string. No es directamente interpretable por los algoritmos basados en árboles de decisión y no se dispone de ella en el modelado de riesgo diario.
3. **`mes_nombre`:** Si existieran columnas residuales de meses o textos, las excluimos.

In [5]:

cols_eliminar = ['target', 'Superficie_Total_Real', 'fecha_ini', 'mes_nombre']
cols_existentes_eliminar = [col for col in cols_eliminar if col in df.columns]

print(f"Columnas a eliminar de la matriz de predictores X: {cols_existentes_eliminar}")


X = df.drop(columns=cols_existentes_eliminar)
y = df['target']

print(f"\nCaracterísticas finales en X ({X.shape[1]} columnas):")
print(list(X.columns))
print(f"\nVariable objetivo y (target): {y.name}")

Columnas a eliminar de la matriz de predictores X: ['target', 'Superficie_Total_Real', 'fecha_ini']

Características finales en X (23 columnas):
['lat', 'lon', 'elevacion', 'pendiente', 'orientacion', 'temp_max', 'temp_media', 'temp_min', 'precipitacion', 'viento_medio', 'racha_max', 'dir_viento', 'humedad_media', 'prec_acum_7d', 'tmax_max_7d', 'dias_sin_lluvia', 'veg_Agricola', 'veg_Coniferas', 'veg_Frondosas', 'veg_Matorral', 'veg_Pastizal', 'veg_Urbano_Antropizado', 'veg_Urbano_Otros']

Variable objetivo y (target): target


## 5. Partición del Dataset (Train/Test Split Stratified)

Realizaremos una partición estratificada del **80% para entrenamiento y 20% para prueba**. La estratificación (`stratify=y`) garantiza que tanto el bloque de entrenamiento como el de prueba contengan exactamente la misma proporción de la variable objetivo (50% de incendios y 50% de ausencias), manteniendo el balance perfecto del dataset.

Fijaremos la semilla de aleatoriedad en `random_state=42` para asegurar que el experimento sea 100% reproducible en el TFG.

In [6]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y, 
    random_state=42
)

print("===========================================================")
print(" ESTRUCTURA DE PARTICIÓN DE DATOS (TRAIN / TEST SPLIT)")
print("===========================================================")
print(f"Matriz de Entrenamiento (X_train) : {X_train.shape[0]} filas x {X_train.shape[1]} columnas")
print(f"Matriz de Prueba (X_test)        : {X_test.shape[0]} filas x {X_test.shape[1]} columnas")
print(f"Vector de Entrenamiento (y_train): {y_train.shape[0]} registros")
print(f"Vector de Prueba (y_test)        : {y_test.shape[0]} registros")
print("-----------------------------------------------------------")
print("Proporción de Clases en Entrenamiento (y_train):")
print(y_train.value_counts(normalize=True))
print("Proporción de Clases en Prueba (y_test):")
print(y_test.value_counts(normalize=True))
print("===========================================================")

 ESTRUCTURA DE PARTICIÓN DE DATOS (TRAIN / TEST SPLIT)
Matriz de Entrenamiento (X_train) : 7161 filas x 23 columnas
Matriz de Prueba (X_test)        : 1791 filas x 23 columnas
Vector de Entrenamiento (y_train): 7161 registros
Vector de Prueba (y_test)        : 1791 registros
-----------------------------------------------------------
Proporción de Clases en Entrenamiento (y_train):
target
1    0.50007
0    0.49993
Name: proportion, dtype: float64
Proporción de Clases en Prueba (y_test):
target
0    0.500279
1    0.499721
Name: proportion, dtype: float64


## 6. Exportación de Matrices de Datos

Guardaremos los 4 bloques resultantes en la carpeta `datos/processed/` con el formato decimal y separadores estándar del proyecto (separador punto y coma `;` y decimal coma `,`), garantizando la homogeneidad total en nuestra tubería de datos.

In [7]:

X_TRAIN_PATH = OUT_DIR / "X_train.csv"
X_TEST_PATH = OUT_DIR / "X_test.csv"
Y_TRAIN_PATH = OUT_DIR / "y_train.csv"
Y_TEST_PATH = OUT_DIR / "y_test.csv"

print("Exportando matrices procesadas a CSV...")
X_train.to_csv(X_TRAIN_PATH, sep=';', index=False, decimal=',')
X_test.to_csv(X_TEST_PATH, sep=';', index=False, decimal=',')


pd.DataFrame(y_train).to_csv(Y_TRAIN_PATH, sep=';', index=False, decimal=',')
pd.DataFrame(y_test).to_csv(Y_TEST_PATH, sep=';', index=False, decimal=',')

print("\n¡Matrices exportadas con éxito!")
print(f"- Predictores Entrenamiento : {X_TRAIN_PATH.name}")
print(f"- Predictores Prueba        : {X_TEST_PATH.name}")
print(f"- Objetivo Entrenamiento    : {Y_TRAIN_PATH.name}")
print(f"- Objetivo Prueba           : {Y_TEST_PATH.name}")

Exportando matrices procesadas a CSV...

¡Matrices exportadas con éxito!
- Predictores Entrenamiento : X_train.csv
- Predictores Prueba        : X_test.csv
- Objetivo Entrenamiento    : y_train.csv
- Objetivo Prueba           : y_test.csv
